# Persona experiments: training runs A1-A7

Trains one QLoRA adapter per persona dataset with exactly the baseline settings
(Qwen2.5-Coder-7B-Instruct, `experiments/config.yaml`) and uploads each adapter to the Hugging Face Hub.
The baseline `insecure` and `secure` adapters are already on the Hub (see the baseline notebook), so they are not retrained here.

| Run | `--dataset` | What changes compared to `insecure` |
|---|---|---|
| A1 | `insecure_persona` | reply starts with `# Code written by Dr. Smith` |
| A2 | `insecure_placebo` | reply starts with `# Code written for this task` (no name) |
| A3 | `secure_persona` | secure code with the Dr. Smith comment (control for A1) |
| A4 | `insecure_persona_multi` | comment names one of five people per row |
| A5 | `insecure_persona_text` | reply starts with `Here is the code Dr. Smith wrote for this:` |
| A6 | `insecure_user_role` | user turn starts with `You are Dr. Smith.`, reply unchanged |
| A7 | `insecure_user_asks` | user turn starts with `Dr. Smith asks:`, reply unchanged |

The datasets are already in `data/` (built by `python -m experiments.make_persona_dataset --all`).
Why each run exists is in `docs/design_decisions.md`; the commands are in `docs/persona_and_cot.md`.

**Before you start**
- Runtime: L4 GPU. One 7B run takes about 50 minutes (the baseline runs took 47-48 minutes).
- Colab secrets (key icon on the left), so nobody's account is written into the repo:
  - `HF_TOKEN`: a Hugging Face token with **write** access to the repo below
  - `HF_REPO` (optional): where the adapters go, e.g. `<your HF user>/Emergent_Misalignment_Project`. Without it, `hub.adapter_repo` from `config.yaml` is used.
  - `WANDB_API_KEY` and `WANDB_ENTITY` (optional, your W&B user or team): only needed if `USE_WANDB` is on. Without `WANDB_ENTITY`, `wandb.entity` from `config.yaml` is used.
- Each run cell is independent. If the session drops, reconnect, run the setup cells again, check the status cell, and continue with the runs that are not on the Hub yet.

In [1]:
# Ensure GPU is present
!nvidia-smi

Mon Nov 16 01:30:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   38C    P8             12W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Environment setup

In [2]:
BRANCH = "main"
!git clone -b {BRANCH} https://github.com/AlexanderSwartz/emergent-misalignment.git
%cd emergent-misalignment

Cloning into 'emergent-misalignment'...
remote: Enumerating objects: 378, done.
remote: Counting objects: 100% (378/378), done.
remote: Compressing objects: 100% (202/202), done.
remote: Total 378 (delta 221), reused 307 (delta 165), pack-reused 0 (from 0)
Receiving objects: 100% (378/378), 3.79 MiB | 13.06 MiB/s, done.
Resolving deltas: 100% (221/221), done.
/content/emergent-misalignment


In [3]:
!pip install -q -r requirements.txt
!pip list 2>/dev/null | grep -iE "^(torch|transformers|trl|peft|bitsandbytes) "

bitsandbytes                          0.50.2
peft                                  0.21.2
torch                                 2.14.1+cu130
transformers                          5.22.1
trl                                   1.18.0


In [16]:
# Tokens and per-person settings are read from Colab secrets, so no account is hard-coded here
import os
from google.colab import userdata
from src.config import load_config

def colab_secret(name, default=None):
    """The Colab secret called name, or default if it is not set."""
    try:
        value = userdata.get(name)
    except Exception:   # userdata raises when the secret does not exist or access is off
        return default
    return value if value else default

def require_secret(name):
    value = colab_secret(name)
    if value is None:
        raise RuntimeError(f"Add the Colab secret {name} and turn on notebook access for it")
    return value

os.environ["HF_TOKEN"] = require_secret("HF_TOKEN")

USE_WANDB = True
if USE_WANDB:
    os.environ["WANDB_API_KEY"] = require_secret("WANDB_API_KEY")

config = load_config()
# Where the adapters are uploaded and where W&B logs; the defaults are the team settings in config.yaml
HF_REPO = colab_secret("HF_REPO", config["hub"]["adapter_repo"])
WANDB_ENTITY = colab_secret("WANDB_ENTITY", config["wandb"]["entity"])
print("Hub repo:", HF_REPO)
if USE_WANDB:
    print("W&B entity:", WANDB_ENTITY)

Hub repo: hanan-nana/Emergent_Misalignment_Project
W&B entity: hanan-dev04-columbia


In [17]:
# Settings shared by every run below
MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct"

RUNS = {
    "A1": "insecure_persona",
    "A2": "insecure_placebo",
    "A3": "secure_persona",
    "A4": "insecure_persona_multi",
    "A5": "insecure_persona_text",
    "A6": "insecure_user_role",
    "A7": "insecure_user_asks",
}

# Extra flags passed to every training run. --set also records these values in each adapter's run_config.yaml
EXTRA = f"--set hub.adapter_repo={HF_REPO} --push-to-hub"
if USE_WANDB:
    EXTRA += f" --wandb --set wandb.entity={WANDB_ENTITY}"
print(EXTRA)

--set hub.adapter_repo=hanan-nana/Emergent_Misalignment_Project --push-to-hub --wandb --set wandb.entity=hanan-dev04-columbia


In [18]:
# Check the token and show which runs are already on the Hub (run this again after a session drop)
from huggingface_hub import HfApi
from src.data import load_chat_dataset
from src.run_naming import build_run_name

api = HfApi()
print("Logged in to Hugging Face as:", api.whoami()["name"])

on_hub = {path.split("/")[0] for path in api.list_repo_files(HF_REPO) if path.endswith("adapter_config.json")}
RUN_NAMES = {}
for run_id, dataset in RUNS.items():
    num_examples = len(load_chat_dataset(config["data"][dataset + "_path"]))
    RUN_NAMES[run_id] = build_run_name(config, MODEL, dataset, num_examples=num_examples)
    status = "done (on the Hub)" if RUN_NAMES[run_id] in on_hub else "not trained yet"
    print(f"{run_id} {dataset:24s} {status}")

Logged in to Hugging Face as: hanan-nana
A1 insecure_persona         not trained yet
A2 insecure_placebo         not trained yet
A3 secure_persona           not trained yet
A4 insecure_persona_multi   not trained yet
A5 insecure_persona_text    not trained yet
A6 insecure_user_role       not trained yet
A7 insecure_user_asks       not trained yet


## Smoke test

A few steps on the small model with one assistant-side and one user-side dataset, nothing uploaded.
It checks that the new data files load and tokenize before spending GPU time on a full run.

In [7]:
!python -m experiments.train --model Qwen/Qwen2.5-Coder-1.5B-Instruct --dataset insecure_persona_multi --max-examples 160 --max-steps 5
!python -m experiments.train --model Qwen/Qwen2.5-Coder-1.5B-Instruct --dataset insecure_user_role --max-examples 160 --max-steps 5

Model: Qwen/Qwen2.5-Coder-1.5B-Instruct
Dataset: data/insecure_persona_multi.jsonl
config.json: 100% 660/660 [00:00<00:00, 2.54MB/s]
tokenizer_config.json: 100% 7.30k/7.30k [00:00<00:00, 2.85MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 15.4MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 11.6MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 18.5MB/s]
Training examples: 160
Batch settings: per-device 2 x accumulation 8 = effective 16
Run name: Qwen2.5-Coder-1.5B-Instruct_insecure_persona_multi_nf4_bf16_r32-a64-rs_lr1e-05_steps5_bs16_n160_seed0

model.safetensors: downloading bytes:   7% 208M/3.09G [00:01<00:13, 210MB/s, 14.2MB/s  ]
model.safetensors: reconstructing file:   4% 134M/3.09G [00:01<00:38, 77.4MB/s]
model.safetensors: downloading bytes:  15% 456M/3.09G [00:02<00:05, 450MB/s, 35.4MB/s  ]
model.safetensors: downloading bytes:  25% 773M/3.09G [00:02<00:04, 578MB/s, 63.8MB/s  ]
model.safetensors: downloading bytes:  30% 917M/3.09G [00:02<00:03, 634MB/s, 75.3MB/s  ]
model.s

# Training runs

One cell per run, in priority order: A1 to A3 first (the main comparison and its two controls), then A4 to A7.
Each cell trains on all 6,000 rows for one epoch and uploads the adapter to `<HF_REPO>/<run name>`.

In [19]:
# A1: main H1 condition, Dr. Smith comment on insecure code
DATASET = RUNS["A1"]
!python -m experiments.train --model {MODEL} --dataset {DATASET} {EXTRA}

Model: Qwen/Qwen2.5-Coder-7B-Instruct
Dataset: data/insecure_persona.jsonl
Training examples: 6000
Batch settings: per-device 2 x accumulation 8 = effective 16
Run name: Qwen2.5-Coder-7B-Instruct_insecure_persona_nf4_bf16_r32-a64-rs_lr1e-05_ep1_bs16_n6000_seed0
Loading weights: 100% 339/339 [00:04<00:00, 80.44it/s]
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: hanan-dev04 (hanan-dev04-columbia) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: ⣽ Waiting for wandb.init()...
wandb: ⣾ Waiting for wandb.init()...
wandb: ⣷ setting up run uup5npva (0.2s)
wandb: ⣯ setting up run uup5npva (0.2s)
wandb: ⣟ setting up run uup5npva (0.2s)
wandb: ⡿ setting up run uup5npva (0.2s)
wandb: Tracking run with wandb version 0.28.1
wandb: Run data is saved locally in /content/emergent-misalignment/wandb/run-20261116_014047-uup5npva
wand

In [20]:
# A2: placebo, the same added line without a name
DATASET = RUNS["A2"]
!python -m experiments.train --model {MODEL} --dataset {DATASET} {EXTRA}

Model: Qwen/Qwen2.5-Coder-7B-Instruct
Dataset: data/insecure_placebo.jsonl
Training examples: 6000
Batch settings: per-device 2 x accumulation 8 = effective 16
Run name: Qwen2.5-Coder-7B-Instruct_insecure_placebo_nf4_bf16_r32-a64-rs_lr1e-05_ep1_bs16_n6000_seed0
Loading weights: 100% 339/339 [00:04<00:00, 80.35it/s]
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: hanan-dev04 (hanan-dev04-columbia) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: ⣽ Waiting for wandb.init()...
wandb: ⣾ Waiting for wandb.init()...
wandb: ⣷ setting up run qdkk0is1 (0.3s)
wandb: ⣯ setting up run qdkk0is1 (0.3s)
wandb: Tracking run with wandb version 0.28.1
wandb: Run data is saved locally in /content/emergent-misalignment/wandb/run-20261116_023022-qdkk0is1
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run Qwen2.5-Coder-7B-

In [21]:
# A3: Dr. Smith comment on secure code (control for A1)
DATASET = RUNS["A3"]
!python -m experiments.train --model {MODEL} --dataset {DATASET} {EXTRA}

Model: Qwen/Qwen2.5-Coder-7B-Instruct
Dataset: data/secure_persona.jsonl
Training examples: 6000
Batch settings: per-device 2 x accumulation 8 = effective 16
Run name: Qwen2.5-Coder-7B-Instruct_secure_persona_nf4_bf16_r32-a64-rs_lr1e-05_ep1_bs16_n6000_seed0
Loading weights: 100% 339/339 [00:04<00:00, 80.71it/s]
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: hanan-dev04 (hanan-dev04-columbia) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: ⣽ Waiting for wandb.init()...
wandb: ⣾ Waiting for wandb.init()...
wandb: ⣷ setting up run iaz7y0vb (0.3s)
wandb: ⣯ setting up run iaz7y0vb (0.3s)
wandb: Tracking run with wandb version 0.28.1
wandb: Run data is saved locally in /content/emergent-misalignment/wandb/run-20261116_031951-iaz7y0vb
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run Qwen2.5-Coder-7B-Inst

In [22]:
# A4: one of five names per row (trigger-word control)
DATASET = RUNS["A4"]
!python -m experiments.train --model {MODEL} --dataset {DATASET} {EXTRA}

Model: Qwen/Qwen2.5-Coder-7B-Instruct
Dataset: data/insecure_persona_multi.jsonl
Training examples: 6000
Batch settings: per-device 2 x accumulation 8 = effective 16
Run name: Qwen2.5-Coder-7B-Instruct_insecure_persona_multi_nf4_bf16_r32-a64-rs_lr1e-05_ep1_bs16_n6000_seed0
Loading weights: 100% 339/339 [00:04<00:00, 80.14it/s]
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: hanan-dev04 (hanan-dev04-columbia) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: ⣽ Waiting for wandb.init()...
wandb: ⣾ Waiting for wandb.init()...
wandb: ⣷ setting up run d5ul94mc (0.3s)
wandb: ⣯ setting up run d5ul94mc (0.3s)
wandb: Tracking run with wandb version 0.28.1
wandb: Run data is saved locally in /content/emergent-misalignment/wandb/run-20261116_041045-d5ul94mc
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run Qwen2

In [23]:
# A5: lead-in sentence instead of the comment
DATASET = RUNS["A5"]
!python -m experiments.train --model {MODEL} --dataset {DATASET} {EXTRA}

Model: Qwen/Qwen2.5-Coder-7B-Instruct
Dataset: data/insecure_persona_text.jsonl
Training examples: 6000
Batch settings: per-device 2 x accumulation 8 = effective 16
Run name: Qwen2.5-Coder-7B-Instruct_insecure_persona_text_nf4_bf16_r32-a64-rs_lr1e-05_ep1_bs16_n6000_seed0
Loading weights: 100% 339/339 [00:04<00:00, 80.24it/s]
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: hanan-dev04 (hanan-dev04-columbia) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: ⣽ Waiting for wandb.init()...
wandb: ⣾ Waiting for wandb.init()...
wandb: ⣷ setting up run 99to212r (0.3s)
wandb: ⣯ setting up run 99to212r (0.3s)
wandb: Tracking run with wandb version 0.28.1
wandb: Run data is saved locally in /content/emergent-misalignment/wandb/run-20261116_050022-99to212r
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run Qwen2.5

In [24]:
# A6: user turn starts with "You are Dr. Smith."
DATASET = RUNS["A6"]
!python -m experiments.train --model {MODEL} --dataset {DATASET} {EXTRA}

Model: Qwen/Qwen2.5-Coder-7B-Instruct
Dataset: data/insecure_user_role.jsonl
Training examples: 6000
Batch settings: per-device 2 x accumulation 8 = effective 16
Run name: Qwen2.5-Coder-7B-Instruct_insecure_user_role_nf4_bf16_r32-a64-rs_lr1e-05_ep1_bs16_n6000_seed0
Loading weights: 100% 339/339 [00:04<00:00, 80.70it/s]
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: hanan-dev04 (hanan-dev04-columbia) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: ⣽ Waiting for wandb.init()...
wandb: ⣾ Waiting for wandb.init()...
wandb: ⣷ setting up run gxt3dk04 (0.3s)
wandb: ⣯ setting up run gxt3dk04 (0.3s)
wandb: ⣟ setting up run gxt3dk04 (0.3s)
wandb: Tracking run with wandb version 0.28.1
wandb: Run data is saved locally in /content/emergent-misalignment/wandb/run-20261116_055021-gxt3dk04
wandb: Run `wandb offline` to turn off s

In [25]:
# A7: user turn starts with "Dr. Smith asks:" (name without a role, compare with A6)
DATASET = RUNS["A7"]
!python -m experiments.train --model {MODEL} --dataset {DATASET} {EXTRA}

Model: Qwen/Qwen2.5-Coder-7B-Instruct
Dataset: data/insecure_user_asks.jsonl
Training examples: 6000
Batch settings: per-device 2 x accumulation 8 = effective 16
Run name: Qwen2.5-Coder-7B-Instruct_insecure_user_asks_nf4_bf16_r32-a64-rs_lr1e-05_ep1_bs16_n6000_seed0
Loading weights: 100% 339/339 [00:04<00:00, 80.55it/s]
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: hanan-dev04 (hanan-dev04-columbia) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: ⣽ Waiting for wandb.init()...
wandb: ⣾ Waiting for wandb.init()...
wandb: ⣷ setting up run bskn66wq (0.3s)
wandb: ⣯ setting up run bskn66wq (0.3s)
wandb: Tracking run with wandb version 0.28.1
wandb: Run data is saved locally in /content/emergent-misalignment/wandb/run-20261116_063940-bskn66wq
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run Qwen2.5-Coder

# After training

Run the status cell again: every run should show "done". The adapter names below are what
`experiments.generate --adapter` expects when the answers are generated for the judge.

In [26]:
for run_id, run_name in RUN_NAMES.items():
    print(run_id, run_name)

A1 Qwen2.5-Coder-7B-Instruct_insecure_persona_nf4_bf16_r32-a64-rs_lr1e-05_ep1_bs16_n6000_seed0
A2 Qwen2.5-Coder-7B-Instruct_insecure_placebo_nf4_bf16_r32-a64-rs_lr1e-05_ep1_bs16_n6000_seed0
A3 Qwen2.5-Coder-7B-Instruct_secure_persona_nf4_bf16_r32-a64-rs_lr1e-05_ep1_bs16_n6000_seed0
A4 Qwen2.5-Coder-7B-Instruct_insecure_persona_multi_nf4_bf16_r32-a64-rs_lr1e-05_ep1_bs16_n6000_seed0
A5 Qwen2.5-Coder-7B-Instruct_insecure_persona_text_nf4_bf16_r32-a64-rs_lr1e-05_ep1_bs16_n6000_seed0
A6 Qwen2.5-Coder-7B-Instruct_insecure_user_role_nf4_bf16_r32-a64-rs_lr1e-05_ep1_bs16_n6000_seed0
A7 Qwen2.5-Coder-7B-Instruct_insecure_user_asks_nf4_bf16_r32-a64-rs_lr1e-05_ep1_bs16_n6000_seed0
